# 02 - SAR-Optical Fusion: Wheat Sown Area Mapping

Monthly **Sentinel-1 VV/VH** + **Sentinel-2 NDVI** composites are stacked and classified with **Random Forest** (GEE smileRandomForest). Outputs: wheat map, sown area in **lakh hectares** per state, state choropleth, and SAR backscatter vs phenology curves.

> Training here uses NDVI-heuristic **pseudo-labels (DEMO)**. Replace with real field-survey / crop-cutting-experiment points for operational accuracy.

In [1]:
import sys, yaml
sys.path.append('..')
import ee, pandas as pd
from src import gee_utils, indices, classification, sar_processing, visualization

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
gee_utils.init_ee()
SEASON_YEAR = 2023
MONTHS = cfg['classification']['monthly_composites']

E:\codes\Gitlab project\wheat-crop-monitoring\venv\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## End-to-end demo on one state (Punjab)
The full 8-state loop is at the bottom (long-running; consider Export tasks).

In [2]:
state = 'Punjab'
aoi_state = gee_utils.get_state_geometry(state)
stack = classification.build_feature_stack(aoi_state, SEASON_YEAR, MONTHS, gee_utils, indices)
print('Feature bands:', stack.bandNames().getInfo())

Feature bands: ['VV_11', 'VH_11', 'NDVI_11', 'VV_12', 'VH_12', 'NDVI_12', 'VV_01', 'VH_01', 'NDVI_01', 'VV_02', 'VH_02', 'NDVI_02', 'VV_03', 'VH_03', 'NDVI_03']


In [3]:
# Pseudo-labels: peak-season (Jan 15 - Feb 28) NDVI > 0.55 -> wheat (DEMO ONLY)
peak_ndvi = (gee_utils.get_s2_collection(aoi_state, f'{SEASON_YEAR + 1}-01-15', f'{SEASON_YEAR + 1}-02-28')
             .map(indices.add_ndvi_s2).select('NDVI').median())
label_img = peak_ndvi.gt(0.55).rename('class')
samples_fc = label_img.stratifiedSample(numPoints=300, classBand='class', region=aoi_state,
                                        scale=20, seed=42, geometries=True).randomColumn(seed=1)
train_fc = samples_fc.filter(ee.Filter.lt('random', cfg['classification']['train_split']))
val_fc = samples_fc.filter(ee.Filter.gte('random', cfg['classification']['train_split']))
print('train:', train_fc.size().getInfo(), '| val:', val_fc.size().getInfo())

train: 420 | val: 180


In [4]:
clf, _ = classification.train_rf(stack, train_fc, 'class', cfg['classification']['n_trees'])
val_samples = stack.sampleRegions(collection=val_fc, properties=['class'], scale=20, tileScale=4)
report = classification.accuracy_report(clf, val_samples, 'class')
print('Overall accuracy:', round(report['overall_accuracy'], 3), '| kappa:', round(report['kappa'], 3))

Overall accuracy: 0.971 | kappa: 0.938


In [5]:
wheat_map = classification.classify(stack, clf)
area = classification.area_lakh_ha(wheat_map, aoi_state, scale=1000).getInfo()
print(f'{state} wheat sown area: {area:.2f} lakh ha')

import geemap
m = geemap.Map()
m.centerObject(aoi_state, 7)
m.addLayer(wheat_map.selfMask(), {'palette': ['gold']}, 'Wheat')
m

EEException: Computation timed out.

## All-states loop -> sown area table + choropleth

In [ ]:
import concurrent.futures

def process_state(st):
    g = gee_utils.get_state_geometry(st)
    stk = classification.build_feature_stack(g, SEASON_YEAR, MONTHS, gee_utils, indices)
    wm = classification.classify(stk, clf)  # operational: retrain per agro-climatic zone
    area = classification.area_lakh_ha(wm, g, scale=1000).getInfo()
    return {'state': st, 'area_lakh_ha': area}

# Run GEE classifications in parallel using ThreadPoolExecutor
print("Starting concurrent GEE classifications for all states...")
with concurrent.futures.ThreadPoolExecutor(max_workers=len(cfg['states'])) as executor:
    rows = list(executor.map(process_state, cfg['states']))

sown_df = pd.DataFrame(rows)
sown_df.to_csv('../outputs/sown_area_estimates.csv', index=False)
print('Total:', round(sown_df.area_lakh_ha.sum(), 1), 'lakh ha')

In [ ]:
import geopandas as gpd
states_fc = gee_utils.get_states_fc(cfg['states'])
gdf = gpd.GeoDataFrame.from_features(states_fc.getInfo()['features'])
gdf['state'] = gdf['ADM1_NAME']
gdf = gdf.merge(sown_df, on='state')
visualization.sown_area_choropleth(gdf)

## SAR backscatter vs wheat phenology
VH backscatter rises with biomass through tillering-heading and drops near maturity/harvest.

In [ ]:
# wheat_zone = aoi_state.intersection(wheat_map.selfMask().geometry(), 1000) # Unused and causes GEE timeout
s1 = gee_utils.get_s1_collection(aoi_state, f'{SEASON_YEAR}-11-01', f'{SEASON_YEAR + 1}-04-30')
ts = sar_processing.backscatter_timeseries(s1.map(sar_processing.speckle_filter), aoi_state, band='VH')
visualization.phenology_backscatter_curve(ts, sar_processing.WHEAT_PHENOLOGY)